In [ ]:
# Libraries and reproducibility

import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from tensorflow.keras.optimizers import Adam

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from imblearn.over_sampling import SMOTE
from matplotlib.patches import Patch
from collections import Counter

: 

In [ ]:
# Fix all randomness so results are identical every run
os.environ['PYTHONHASHSEED'] = '42'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f"Libraries loaded | TensorFlow: {tf.__version__}")

In [ ]:
# Load datasets
train_df = pd.read_csv("/content/drive/MyDrive/DSPL_clean_datasets/train_cleaned.csv")
val_df   = pd.read_csv("/content/drive/MyDrive/DSPL_clean_datasets/validation_cleaned.csv")
test_df  = pd.read_csv("/content/drive/MyDrive/DSPL_clean_datasets/test_cleaned.csv")

print(f"Train      : {len(train_df):,} rows")
print(f"Validation : {len(val_df):,} rows")
print(f"Test       : {len(test_df):,} rows (no labels — predictions only)")

In [ ]:
# Fix distribution mismatch + stratified re-split
# Problem: train had 23% cancellation rate, validation had 41%.
# Training on one distribution and evaluating on another caused all models to score ~50% F1 (essentially random guessing).
# Fix: combine both labelled datasets, then re-split 80/20
# using stratified sampling so both splits share the same 24.7% cancellation rate.

labelled = pd.concat([train_df, val_df], ignore_index=True)

train_split, val_split = train_test_split(
    labelled,
    test_size   = 0.20,
    random_state = 42,
    stratify    = labelled["Reservation_Status"]
)
train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)

print(f"\nAfter stratified re-split:")
print(f"  Train : {len(train_split):,} rows")
print(f"  Val   : {len(val_split):,} rows")
for name, df in [("Train", train_split), ("Val", val_split)]:
    rates = " | ".join([
        f"{s}={(df['Reservation_Status']==s).mean():.1%}"
        for s in ["Check-Out", "Canceled", "No-Show"]
    ])
    print(f"  {name}: {rates}")
print(" Both splits now have matching class distributions")

In [ ]:
# Feature engineering
# Eight new features derived from date columns.
# These capture patterns the raw date values cannot.

def engineer_features(df):
    df = df.copy()
    for col in ["Booking_date", "Expected_checkin", "Expected_checkout"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["lead_time"]       = (df["Expected_checkin"] - df["Booking_date"]).dt.days.clip(lower=0)
    df["length_of_stay"]  = (df["Expected_checkout"] - df["Expected_checkin"]).dt.days.clip(lower=1)
    df["total_guests"]    = df["Adults"] + df["Children"] + df["Babies"]
    df["checkin_month"]   = df["Expected_checkin"].dt.month
    df["checkin_dow"]     = df["Expected_checkin"].dt.dayofweek   # 0=Mon … 6=Sun
    df["is_weekend"]      = df["checkin_dow"].isin([5, 6]).astype(int)
    df["revenue_at_risk"] = df["Room_Rate"] * df["length_of_stay"]
    df["booking_month"]   = df["Booking_date"].dt.month
    return df

train_split = engineer_features(train_split)
val_split   = engineer_features(val_split)
test_df     = engineer_features(test_df)
print("Features engineered: lead_time, length_of_stay, total_guests,")
print("checkin_month, checkin_dow, is_weekend, revenue_at_risk, booking_month")

In [ ]:
# Define feature sets
NUMERIC_FEATURES = [
    "lead_time", "Room_Rate", "Age", "checkin_month", "checkin_dow",
    "Discount_Rate", "total_guests", "length_of_stay", "Adults",
    "Children", "Babies", "is_weekend", "revenue_at_risk", "booking_month",
]
CATEGORICAL_FEATURES = [
    "Meal_Type", "Booking_channel", "Deposit_type", "Hotel_Type",
    "Country_region", "Visted_Previously", "Previous_Cancellations",
    "Gender", "Required_Car_Parking", "Use_Promotion",
]

# Keep only columns that exist in all 3 datasets
NUMERIC_FEATURES     = [f for f in NUMERIC_FEATURES
                         if f in train_split.columns and f in test_df.columns]
CATEGORICAL_FEATURES = [f for f in CATEGORICAL_FEATURES
                         if f in train_split.columns and f in test_df.columns]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print(f"\nFeatures selected: {len(ALL_FEATURES)} total")
print(f"  Numeric     ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"  Categorical ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")

In [ ]:
# One-hot encode categorical features
# Fit get_dummies on all 3 datasets combined so every dataset gets identical column structure after encoding.

combined_raw = pd.concat([
    train_split[ALL_FEATURES],
    val_split[ALL_FEATURES],
    test_df[ALL_FEATURES],
], ignore_index=True)

combined_enc = pd.get_dummies(combined_raw, columns=CATEGORICAL_FEATURES, drop_first=False)
combined_enc.fillna(combined_enc.median(numeric_only=True), inplace=True)

n_tr = len(train_split)
n_v  = len(val_split)

X_train_enc   = combined_enc.iloc[:n_tr].values
X_val_enc     = combined_enc.iloc[n_tr : n_tr + n_v].values
X_test_enc    = combined_enc.iloc[n_tr + n_v:].values
FEATURE_NAMES = combined_enc.columns.tolist()

y_train_raw = train_split["Reservation_Status"].values
y_val_str   = val_split["Reservation_Status"].values

In [ ]:
# Label encoder for Neural Network (requires integer labels)
le = LabelEncoder()
le.fit(["Canceled", "Check-Out", "No-Show"])
y_val_int = le.transform(y_val_str)

print(f"\nEncoded shapes:")
print(f"  X_train : {X_train_enc.shape}")
print(f"  X_val   : {X_val_enc.shape}")
print(f"  X_test  : {X_test_enc.shape}")
print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# SMOTE oversampling (training set only)
# Problem: training set has 3:1 imbalance (75% Check-Out, 16% Canceled, 8% No-Show). Models trained on imbalanced data learn to ignore minority classes.
# Fix: SMOTE creates synthetic minority class rows by interpolating between existing examples, bringing all three classes to equal size.

print(f"\nClass balance before SMOTE: {Counter(y_train_raw)}")

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm_raw = smote.fit_resample(X_train_enc, y_train_raw)

# Force clean string labels after SMOTE
y_train_sm_str = np.array([str(y) for y in y_train_sm_raw])
y_train_sm_int = le.transform(y_train_sm_str)

print(f"Class balance after SMOTE : {Counter(y_train_sm_str)}")
print(f"Training rows: {len(y_train_raw):,} → {len(y_train_sm_str):,}")
print(" All 3 classes now equal in training set")

In [ ]:
# StandardScaler (fit on SMOTE training set only)
# scaler is fitted on SMOTE training data only.
# Val and test are transformed using the same scaler but never used to fit it. This prevents data leakage.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_val_scaled   = scaler.transform(X_val_enc)
X_test_scaled  = scaler.transform(X_test_enc)
print("Scaling complete — fit on SMOTE train, transform val and test")

In [ ]:
# Train Logistic Regression and Decision Tree
sklearn_models = {
    "Logistic Regression": LogisticRegression(
        max_iter     = 3000,
        C            = 0.5,       # stronger regularisation
        solver       = "lbfgs",
        multi_class  = "multinomial",
        random_state = 42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth        = 8,
        min_samples_leaf = 15,
        min_samples_split = 30,
        random_state     = 42
    ),
}

print("\nTraining sklearn models...")
for name, mdl in sklearn_models.items():
    mdl.fit(X_train_scaled, y_train_sm_str)
    classes = sorted(set(mdl.predict(X_val_scaled)))
    ok      = "successful" if "No-Show" in classes else " No-Show missing"
    print(f"  {name:<22} → {classes} {ok}")

In [ ]:
# Build Neural Network
# Architecture: Input → 256 → 128 → 64 → 32 → 3 (softmax)

# Design choices:
#   ReLU hidden layers  — standard for deep networks
#   BatchNormalisation  — stabilises training
#   Dropout             — prevents overfitting
#   L2 regularisation   — penalises large weights
#   Softmax output      — probabilities always sum to exactly 100%
#                         (sigmoid would allow >100% total)
#   Adam optimiser      — adaptive learning rate
#   sparse_categorical_crossentropy — correct loss for integer labels

n_features = X_train_scaled.shape[1]

def build_nn(learning_rate=0.001, dropout_rate=0.3):
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),

        layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),

        layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),

        layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate / 2),

        layers.Dense(32, activation="relu"),
        layers.Dropout(dropout_rate / 2),

        layers.Dense(3, activation="softmax"),  # 3 output classes
    ])
    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = "sparse_categorical_crossentropy",
        metrics   = ["accuracy"],
    )
    return model

nn_model = build_nn(learning_rate=0.001, dropout_rate=0.3)
nn_model.summary()

In [ ]:
# Train Neural Network
# Use validation set for tuning (monitor val_loss)
# Train for a long time (500 epochs maximum)
# EarlyStopping restores the best weights automatically
# ReduceLROnPlateau halves the learning rate when stuck

early_stop = callbacks.EarlyStopping(
    monitor         = "val_loss",
    patience        = 30,
    restore_best_weights = True,
    verbose         = 1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor  = "val_loss",
    factor   = 0.5,
    patience = 10,
    min_lr   = 1e-6,
    verbose  = 1
)

print("\nTraining Neural Network (up to 500 epochs, early stopping on val_loss)...")

history = nn_model.fit(
    X_train_scaled, y_train_sm_int,
    validation_data = (X_val_scaled, y_val_int),
    epochs          = 500,
    batch_size      = 256,
    callbacks       = [early_stop, reduce_lr],
    verbose         = 1
)

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"\n Best epoch: {best_epoch} | Training stopped at epoch: {len(history.history['loss'])}")

In [ ]:
# Verify NN predicts all 3 classes
nn_probs    = nn_model.predict(X_val_scaled, verbose=0)
nn_pred_int = np.argmax(nn_probs, axis=1)
nn_pred_str = le.inverse_transform(nn_pred_int)
print(f"NN classes predicted: {sorted(set(nn_pred_str))} successful")
print(f"Softmax check — probs sum: Mean={nn_probs.sum(axis=1).mean():.4f} successful")

In [ ]:
# Training history plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (train_key, val_key), title in zip(
    axes,
    [("loss", "val_loss"), ("accuracy", "val_accuracy")],
    ["Loss over Epochs", "Accuracy over Epochs"]
):
    ax.plot(history.history[train_key], label="Train", color="#e74c3c")
    ax.plot(history.history[val_key],   label="Val",   color="#3498db")
    ax.axvline(best_epoch - 1, color="green", linestyle="--",
               label=f"Best epoch ({best_epoch})")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle(f"Neural Network Training History | Best epoch: {best_epoch}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("nn_training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to nn_training_history.png")

In [ ]:
# Evaluate all 3 models on validation set
def evaluate(y_true, y_pred):
    return {
        "Accuracy":      accuracy_score(y_true, y_pred),
        "Prec_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Rec_weighted":  recall_score(y_true,    y_pred, average="weighted", zero_division=0),
        "F1_weighted":   f1_score(y_true,        y_pred, average="weighted", zero_division=0),
        "Prec_macro":    precision_score(y_true, y_pred, average="macro",    zero_division=0),
        "Rec_macro":     recall_score(y_true,    y_pred, average="macro",    zero_division=0),
        "F1_macro":      f1_score(y_true,        y_pred, average="macro",    zero_division=0),
    }

all_results = []

for name, mdl in sklearn_models.items():
    r = evaluate(y_val_str, mdl.predict(X_val_scaled))
    r["Model"] = name
    all_results.append(r)

r_nn = evaluate(y_val_str, nn_pred_str)
r_nn["Model"] = "Neural Network"
all_results.append(r_nn)

best_name = max(all_results, key=lambda r: r["F1_weighted"])["Model"]

print(f"\n{'Model':<22} {'Acc':>6}  {'P(W)':>6} {'R(W)':>6} {'F1(W)':>6}  "
      f"{'P(M)':>6} {'R(M)':>6} {'F1(M)':>6}")
print("-" * 80)
for r in all_results:
    flag = " ← BEST" if r["Model"] == best_name else ""
    print(f"{r['Model']:<22} {r['Accuracy']*100:>5.1f}%  "
          f"{r['Prec_weighted']*100:>5.1f}% {r['Rec_weighted']*100:>5.1f}% "
          f"{r['F1_weighted']*100:>5.1f}%  "
          f"{r['Prec_macro']*100:>5.1f}% {r['Rec_macro']*100:>5.1f}% "
          f"{r['F1_macro']*100:>5.1f}%{flag}")
print(f"\n Best model: {best_name}")

In [ ]:
# Model comparison table plot
fig, ax = plt.subplots(figsize=(15, 3.2))
ax.axis("off")

col_labels = [
    "Model", "Accuracy",
    "Precision\n(Weighted)", "Recall\n(Weighted)", "F1\n(Weighted)",
    "Precision\n(Macro)",    "Recall\n(Macro)",    "F1\n(Macro)",
]
table_data = [
    [r["Model"],
     f"{r['Accuracy']*100:.1f}%",
     f"{r['Prec_weighted']*100:.1f}%",
     f"{r['Rec_weighted']*100:.1f}%",
     f"{r['F1_weighted']*100:.1f}%",
     f"{r['Prec_macro']*100:.1f}%",
     f"{r['Rec_macro']*100:.1f}%",
     f"{r['F1_macro']*100:.1f}%"]
    for r in all_results
]

tbl = ax.table(cellText=table_data, colLabels=col_labels,
               cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 2.4)

for j in range(len(col_labels)):
    tbl[0, j].set_facecolor("#2c3e50")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

best_idx = [r["Model"] for r in all_results].index(best_name)
for j in range(len(col_labels)):
    tbl[best_idx + 1, j].set_facecolor("#d5f5e3")
    tbl[best_idx + 1, j].set_text_props(fontweight="bold")

ax.set_title(
    "Model Comparison – 3-Class: Check-Out | Canceled | No-Show\n"
    "Validation Set  |  (W) = Weighted  |  (M) = Macro  |  Green = Best Weighted F1",
    fontsize=10, fontweight="bold", pad=20
)
plt.tight_layout()
plt.savefig("model_comparison_table.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → model_comparison_table.png")

In [ ]:
# Classification report (best model)
best_pred = nn_pred_str if best_name == "Neural Network" \
            else sklearn_models[best_name].predict(X_val_scaled)

print(f"\n{'='*55}")
print(f"  CLASSIFICATION REPORT – {best_name}")
print(f"  Validation set | 3-Class: Check-Out | Canceled | No-Show")
print(f"{'='*55}")
print(classification_report(y_val_str, best_pred, digits=3, zero_division=0))

In [ ]:
# Confusion matrices
CLASS_ORDER = ["Check-Out", "Canceled", "No-Show"]

all_preds = {
    name: (nn_pred_str if name == "Neural Network"
           else sklearn_models[name].predict(X_val_scaled))
    for r in all_results for name in [r["Model"]]
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, preds) in zip(axes, all_preds.items()):
    cm   = confusion_matrix(y_val_str, preds, labels=CLASS_ORDER)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    r    = next(x for x in all_results if x["Model"] == name)
    star = " ⭐" if name == best_name else ""
    ax.set_title(
        f"{name}{star}\n"
        f"Acc={r['Accuracy']*100:.1f}%  "
        f"P(W)={r['Prec_weighted']*100:.1f}%  "
        f"F1(W)={r['F1_weighted']*100:.1f}%",
        fontsize=9, fontweight="bold"
    )
    ax.tick_params(axis="x", rotation=20)

fig.suptitle(
    "Confusion Matrices – Validation Set (All 3 Models)\n"
    "3-Class: Check-Out | Canceled | No-Show",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig("confusion_matrices_3models.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to confusion_matrices_3models.png")

In [ ]:
# Neural Network feature importance
# Neural Networks do not have built-in feature importance.
# Permutation Importance is used

print("\nCalculating Neural Network permutation importance...")
print("(Each feature shuffled 10 times — takes ~2 minutes)")

def nn_scorer(X, y_int):
    probs    = nn_model.predict(X, verbose=0)
    pred_int = np.argmax(probs, axis=1)
    return f1_score(y_int, pred_int, average="macro", zero_division=0)

baseline_f1  = nn_scorer(X_val_scaled, y_val_int)
n_feats      = X_val_scaled.shape[1]
n_repeats    = 10
imp_matrix   = np.zeros((n_feats, n_repeats))

for fi in range(n_feats):
    for rep in range(n_repeats):
        X_perm = X_val_scaled.copy()
        np.random.seed(rep)
        np.random.shuffle(X_perm[:, fi])
        imp_matrix[fi, rep] = baseline_f1 - nn_scorer(X_perm, y_val_int)
    if fi % 10 == 0:
        print(f"  {fi}/{n_feats} features done...")

imp_df = pd.DataFrame({
    "Feature":    FEATURE_NAMES,
    "Importance": imp_matrix.mean(axis=1),
    "Std":        imp_matrix.std(axis=1),
}).sort_values("Importance", ascending=False).reset_index(drop=True)

print(" Permutation importance complete")
print(f"\nTop 15 features – Neural Network:")
print(f"{'Rank':<5} {'Feature':<35} {'Importance':>12} {'Std':>8}")
print("-" * 60)
for i, row in imp_df.head(15).iterrows():
    print(f"{i+1:<5} {row['Feature']:<35} {row['Importance']:>12.4f} ±{row['Std']:>6.4f}")

In [ ]:
# Plot
top15 = imp_df.head(15).copy()
fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ["#e74c3c" if i < 5 else "#3498db" for i in range(len(top15))]
ax.barh(range(len(top15)), top15["Importance"],
        xerr=top15["Std"], color=bar_colors, edgecolor="white",
        capsize=4, error_kw={"elinewidth": 1, "ecolor": "gray"})
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15["Feature"], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel("Drop in Macro F1 when feature is shuffled", fontsize=11)
ax.set_title(
    "Neural Network Feature Importance (Permutation Method)\n"
    "Larger bar = model relies on this feature more",
    fontsize=13, fontweight="bold"
)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.legend(handles=[
    Patch(color="#e74c3c", label="Top 5 most important"),
    Patch(color="#3498db", label="Other features"),
])
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("nn_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to nn_feature_importance.png")

In [ ]:
# Test set predictions
# The test set has no labels — the best model generates a cancellation probability score for each of the 4,291 bookings.
# Three probabilities are output per booking:
#   P(Check-Out)% — probability of completing the stay
#   P(Canceled)%  — probability of canceling in advance
#   P(No-Show)%   — probability of not arriving without notice
# Risk Tier = High if P(Cancel) + P(No-Show) >= 60%

if best_name == "Neural Network":
    test_probs = nn_model.predict(X_test_scaled, verbose=0)
    test_preds = le.inverse_transform(np.argmax(test_probs, axis=1))
    cls_order  = list(le.classes_)
else:
    bm         = sklearn_models[best_name]
    test_probs = bm.predict_proba(X_test_scaled)
    test_preds = bm.predict(X_test_scaled)
    cls_order  = list(bm.classes_)

ci        = cls_order.index("Canceled")
ni        = cls_order.index("No-Show")
risk_prob = test_probs[:, ci] + test_probs[:, ni]

output = test_df[[
    "Reservation-id", "Hotel_Type", "Deposit_type", "lead_time", "Room_Rate"
]].copy().reset_index(drop=True)

for i, cls in enumerate(cls_order):
    output[f"P({cls})%"] = (test_probs[:, i] * 100).round(1)

output["Predicted_Status"] = test_preds
output["Risk_Tier"]        = [
    "High" if p >= 0.60 else "Medium" if p >= 0.30 else "Low"
    for p in risk_prob
]

output = output.sort_values(
    "Risk_Tier",
    key=lambda s: s.map({"High": 0, "Medium": 1, "Low": 2})
).reset_index(drop=True)
output.index += 1

output.to_csv("test_predictions_final.csv", index_label="Rank")
print("\n Saved → test_predictions_final.csv")
print(f"\nRisk tier breakdown ({len(output):,} test bookings):")
print(output["Risk_Tier"].value_counts())
print(f"\nTop 10 highest-risk bookings:")
print(output.head(10).to_string())